In [ ]:
%jars pg/pg-api/build/libs/pg-api-1.0.0-SNAPSHOT.jar 
%jars pg/pg-global/build/libs/pg-global-1.0.0-SNAPSHOT.jar
%jars pg/pg-multiverse/build/libs/pg-multiverse-1.0.0-SNAPSHOT.jar
%jars pg/pg-io/build/libs/pg-io-1.0.0-SNAPSHOT.jar
%jars pgv/examples/pgv-exporter/target/pgv-exporter-0.1.0-SNAPSHOT.jar

In [ ]:
import java.io.File;
import java.io.IOException;
import java.nio.channels.FileChannel;
import java.nio.file.StandardOpenOption;

import dev.chpg.pg.api.AttributeValue;
import dev.chpg.pg.global.GlobalGraph;
import dev.chpg.pg.io.DirectGraphBufferReader;

In [ ]:
long start = System.currentTimeMillis();
File file = new File(new File("data"), "xinu.dgb");
GlobalGraph targetGraph = new GlobalGraph();
try (FileChannel channel = FileChannel.open(file.toPath(), StandardOpenOption.READ)) {
    DirectGraphBufferReader.read(channel, targetGraph, targetGraph.factory(), targetGraph.factory());
}
long stop = System.currentTimeMillis();
System.out.println("Time: " + (stop-start));

System.out.println("Nodes: " + targetGraph.nodes().size());
System.out.println("Edges: " + targetGraph.edges().size());

System.out.println(
    targetGraph.nodes()
        .withAnyTag("XCSG.Function")
        .withAttribute("XCSG.name", AttributeValue.value("freebuf"))
        .toString()
);

System.out.println(
    targetGraph.nodes()
        .withAnyTag("XCSG.Function")
        .withAttribute("XCSG.name", AttributeValue.value("freebuf"))
        .one().get().tags().toString()
);

System.out.println(
    targetGraph.edges()
        .withAnyTag("XCSG.Call")
        .size()
);

In [ ]:
import dev.chpg.pg.multiverse.universe.*;
import dev.pgv.exporter.*; 
import java.io.ByteArrayOutputStream;
import java.nio.charset.StandardCharsets;
import java.util.List;
import java.util.Map;

// 1. Create the backend Universe graph
Universe universe = new Universe();
int nodeId = universe.idGenerator().createNodeId();
UniverseNode helloNode = new UniverseNode(universe, nodeId);
helloNode.attributes().put("name", "hello");

// 2. Minimal manual adapter to Pgv Export Records
record GraphNode(String id, List<String> tags, Map<String, Object> attributes) implements ExportNode {}
record GraphEdge(String id, String source, String target, List<String> tags, Map<String, Object> attributes) implements ExportEdge {}
record GraphSnapshot(String graphId, long version, List<GraphNode> nodesList, List<GraphEdge> edgesList) implements ExportGraph {
    public ExportSchema schema() { return null; }
    public Iterable<? extends ExportNode> nodes() { return nodesList; }
    public Iterable<? extends ExportEdge> edges() { return edgesList; }
}

// Map the Universe node
GraphNode exportHelloNode = new GraphNode(
    String.valueOf(helloNode.id()), 
    List.of("TestNode"), 
    Map.of("name", helloNode.attributes().get("name").toString())
);

GraphSnapshot snapshot = new GraphSnapshot("hello-world-graph", 1, List.of(exportHelloNode), List.of());

// 3. Serialize to JSON string
ByteArrayOutputStream baos = new ByteArrayOutputStream();
PgvExporter exporter = new PgvExporter();
exporter.exportGraph(snapshot, baos);
String jsonPayload = baos.toString(StandardCharsets.UTF_8);

// 4. Generate unique ID and HTML payload
String containerId = "pgv-viz-" + System.currentTimeMillis();
String html = String.format("""
    <div id="%s" style="width: 100%%; height: 600px; border: 1px solid #ccc; border-radius: 8px;"></div>
    <script src="/files/pgv/dist/pgv-bundle.js"></script>
    <script>
        setTimeout(() => {
            if(window.renderPgvNotebook) {
                window.renderPgvNotebook("%s", %%s);
            } else {
                document.getElementById("%s").innerHTML = "Error: pgv-bundle.js not loaded.";
            }
        }, 100);
    </script>
    """, containerId, containerId, containerId);

// Inject the JSON safely
html = String.format(html, jsonPayload);

// 5. Inject into Jupyter output
display("text/html", html);
